In [3]:
# This file includes the codes in pdb_manipulation.ipynb file as functions
# Functions take ppdb_codes such as 2C0K as input 

import sys
from biopandas.pdb import PandasPdb
import numpy as np
import os

# ppdb = PandasPdb().fetch_pdb('2C0K')

In [4]:
# get_sequence function extracts the sequence from the ATOM section in pdb data file
def get_sequence(ppdb):
    amino_acid_sequence = ppdb.amino3to1().residue_name
    amino_acid_sequence = ''.join(amino_acid_sequence)
    return amino_acid_sequence

In [5]:
# get_seqres_sequence function extracts amino acids from the SEQRES section, converts them from 3 letter code to 1 letter code and returns the sequence
def get_seqres_sequence(ppdb):
    df_others = ppdb.df["OTHERS"]
    seqres_entry = df_others[df_others['record_name'] == 'SEQRES']['entry'].values

    # Initialize an empty dictionary
    result_dict = {}

    # Process each string in the array
    for item in seqres_entry:
        # Split the string into words
        words = item.split()
        
        # Extract the second letter to get the chain identifier "A"
        second_letter = words[1]
        
        # Extract the substring after the '46', which starts from the 4th word
        substring = ' '.join(words[3:])
        
        # Append the substring to the corresponding key in the dictionary
        if second_letter in result_dict:
            result_dict[second_letter].append(substring)
        else:
            result_dict[second_letter] = [substring]

    # print(result_dict)

    # Initialize a new dictionary to store the merged strings
    merged_dict = {}

    # Iterate through each key in the dictionary
    for key, strings in result_dict.items():
        # Merge all strings in the list into one string
        merged_string = ' '.join(strings)
        # Add the merged string to the new dictionary
        merged_dict[key] = merged_string

    # print(merged_dict)

    # Mapping dictionary from three-letter to one-letter amino acid codes
    three_to_one = {
        "ALA": "A", "ARG": "R", "ASN": "N", "ASP": "D", "CYS": "C",
        "GLU": "E", "GLN": "Q", "GLY": "G", "HIS": "H", "ILE": "I",
        "LEU": "L", "LYS": "K", "MET": "M", "PHE": "F", "PRO": "P",
        "SER": "S", "THR": "T", "TRP": "W", "TYR": "Y", "VAL": "V", "SEC": "U"
    }

    # Function to convert three-letter codes to one-letter codes
    def convert_to_one_letter(amino_acid_sequence):
        one_letter_sequence = ''.join(three_to_one[aa] for aa in amino_acid_sequence.split())
        return one_letter_sequence

    # Convert the sequences in the dictionary
    converted_amino_acids = {key: convert_to_one_letter(seq) for key, seq in merged_dict.items()}

    # Print the converted sequences
    for key, seq in converted_amino_acids.items():
        print(f"{key}: {seq}")

    # Store the converted sequences in strings
    # sequence_A = converted_amino_acids['A']
    # sequence_B = converted_amino_acids['B']

    merged_sequence = "" 

    for chain in converted_amino_acids:
        merged_sequence += converted_amino_acids[chain]

    return merged_sequence



In [6]:
# get_each_chain_sequence function extracts each chain and their sequences
def get_each_chain_sequence(ppdb):
    sequence = ppdb.amino3to1()
    # print(sequence)

    for chain_id in sequence['chain_id'].unique():
        print('Chain ID: %s' % chain_id)
        print(''.join(sequence.loc[sequence['chain_id'] == chain_id, 'residue_name']))
        print("\n")
        

In [ ]:
if len(sys.argv) != 2:
    print("Usage: python script.py <PDB_code>")
    sys.exit(1)

# Get the PDB code from command-line argument
ppdb_code = sys.argv[1]
ppdb = PandasPdb().fetch_pdb(ppdb_code)

# Call the functions with the provided PDB code
print("PDB code: ", ppdb_code)

sequence = get_sequence(ppdb_code)
print(f"Sequence from ATOM section: \t {sequence}")

seqres_sequence = get_seqres_sequence(ppdb_code)
print(f"Sequence from SEQRES section: \t {seqres_sequence}")


print("Sequence for each chain: ")
get_each_chain_sequence(ppdb_code)